# Wattson: serial baseline vs vLLM on a Tesla T4

Same GPU, same model (Qwen2.5-0.5B-Instruct, FP16), same benchmark, two servers:

1. **Baseline**: Hugging Face `transformers`, one request at a time.
2. **vLLM 0.18.0**: continuous batching.

Wattson reads real GPU energy through NVML and splits it between requests,
using each server's queue metrics so waiting requests aren't overcharged.

**Before running:** Accelerator → **GPU T4 x2**, Internet → **On**. Run cells
top to bottom. Total time is about 20 minutes.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=index,name,power.draw,power.limit,memory.total --format=csv

## 2. Install Rust and build Wattson (about 2 minutes)

In [ ]:
%%bash
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null 2>&1
cd /kaggle/working
rm -rf wattson && git clone -q https://github.com/Shub1806/wattson.git
cd wattson && $HOME/.cargo/bin/cargo build --release 2>&1 | tail -2

## 3. Helpers to start and stop servers

In [ ]:
import os, subprocess, time, urllib.request

os.chdir("/kaggle/working/wattson")
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# Same GPU numbering for CUDA and NVML; the model only sees GPU 0.
BASE_ENV = dict(os.environ, CUDA_DEVICE_ORDER="PCI_BUS_ID")
GPU0_ENV = dict(BASE_ENV, CUDA_VISIBLE_DEVICES="0")

def wait_for(url, proc, log, timeout=900):
    start = time.time()
    while time.time() - start < timeout:
        if proc.poll() is not None:
            print(open(log).read()[-4000:])
            raise RuntimeError(f"process exited, see {log}")
        try:
            urllib.request.urlopen(url, timeout=2)
            return
        except Exception:
            time.sleep(3)
    print(open(log).read()[-4000:])
    raise TimeoutError(url)

def start(cmd, env, log):
    return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def start_gateway(log):
    gw = start(["./target/release/wattson"],
               dict(BASE_ENV, WATTSON_ENERGY="nvml", WATTSON_GPU_INDEX="0"), log)
    wait_for("http://127.0.0.1:8080/health", gw, log, timeout=30)
    time.sleep(1)  # give the queue-metrics poller a moment
    print(open(log).read())
    return gw

def stop(*procs):
    for p in procs:
        p.terminate()
    for p in procs:
        p.wait(timeout=60)
    time.sleep(5)  # let the GPU settle back to idle

print("helpers ready")

## 4. Baseline: Hugging Face, one request at a time

Runs before installing vLLM, so vLLM's packages can't affect it.

In [ ]:
hf = start(["python", "-u", "scripts/hf_server.py"],
           dict(GPU0_ENV, MODEL=MODEL, PORT="8000"), "hf_server.log")
wait_for("http://127.0.0.1:8000/health", hf, "hf_server.log")
gw = start_gateway("wattson_hf.log")

In [ ]:
!python scripts/load_test.py --label hf-t4 --model {MODEL} --concurrency 1,2,4 --waves 3 --max-tokens 128

In [ ]:
stop(gw, hf)
print("baseline stopped")

## 5. Install vLLM 0.18.0 (about 5 minutes)

In [ ]:
!pip install -q vllm==0.18.0 2>&1 | tail -3
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

## 6. vLLM

T4 settings: FP16 (no bfloat16 on T4), Triton attention (FlashAttention
needs a newer GPU), eager mode for reliability. Model loading takes a few
minutes the first time.

In [ ]:
import glob

# FlashInfer (used by parts of vLLM) compiles CUDA kernels on first use and
# links against "libcuda.so". Kaggle only ships "libcuda.so.1", so add a symlink.
candidates = []
for d in ["/usr/lib/x86_64-linux-gnu", "/usr/local/nvidia/lib64", "/usr/lib64", "/usr/local/cuda/compat"]:
    candidates += glob.glob(f"{d}/libcuda.so.1")
os.makedirs("/tmp/cudalib", exist_ok=True)
if candidates and not os.path.exists("/tmp/cudalib/libcuda.so"):
    os.symlink(candidates[0], "/tmp/cudalib/libcuda.so")
print("libcuda:", candidates[:1])

# vLLM 0.18 ignores the old VLLM_ATTENTION_BACKEND variable; use the flag.
vllm_env = dict(GPU0_ENV, LIBRARY_PATH="/tmp/cudalib", VLLM_USE_FLASHINFER_SAMPLER="0")
vllm = start(["vllm", "serve", MODEL,
              "--dtype", "half", "--enforce-eager",
              "--attention-backend", "TRITON_ATTN",
              "--max-model-len", "2048", "--gpu-memory-utilization", "0.7",
              "--host", "127.0.0.1", "--port", "8000"],
             vllm_env, "vllm.log")
wait_for("http://127.0.0.1:8000/health", vllm, "vllm.log")
print("vLLM ready")
gw = start_gateway("wattson_vllm.log")

In [ ]:
!python scripts/load_test.py --label vllm-t4 --model {MODEL} --concurrency 1,2,4,8,16 --waves 3 --max-tokens 128

In [ ]:
stop(gw, vllm)
print("vLLM stopped")

## 7. Side-by-side comparison

In [ ]:
import csv
from collections import defaultdict

def summarize(label):
    rows = [r for r in csv.DictReader(open(f"results/{label}.csv")) if int(r["completion_tokens"]) > 0]
    by_c = defaultdict(list)
    for r in rows:
        by_c[int(r["concurrency"])].append(r)
    out = {}
    for c, rs in by_c.items():
        tokens = sum(int(r["completion_tokens"]) for r in rs)
        out[c] = (sum(float(r["energy_j"]) for r in rs) / tokens,
                  sum(float(r["ttft_ms"] or 0) for r in rs) / len(rs))
    return out

hf_s, vl_s = summarize("hf-t4"), summarize("vllm-t4")
print(f"{'conc':>4} | {'HF J/token':>10} {'HF TTFT':>9} | {'vLLM J/token':>12} {'vLLM TTFT':>9} | {'saving':>6}")
for c in sorted(set(hf_s) | set(vl_s)):
    h, v = hf_s.get(c), vl_s.get(c)
    hj = f"{h[0]:.3f}" if h else "-"
    ht = f"{h[1]:.0f} ms" if h else "-"
    vj = f"{v[0]:.3f}" if v else "-"
    vt = f"{v[1]:.0f} ms" if v else "-"
    save = f"{h[0] / v[0]:.1f}x" if h and v else ""
    print(f"{c:>4} | {hj:>10} {ht:>9} | {vj:>12} {vt:>9} | {save:>6}")

## 8. Download results

From the **Output** panel, download `wattson/results/hf-t4.csv` and
`wattson/results/vllm-t4.csv` and commit them to your repo's `results/`
folder. Then stop the session (power icon) to save GPU hours.